In [ ]:
import os
import pandas as pd
from openpyxl import Workbook
from openpyxl.chart import ScatterChart, Series, Reference
from openpyxl.utils.dataframe import dataframe_to_rows

# 🔧 Endre denne banen til din lokale mappe
mappebane = r"\\tos-nasuni-01\GEO\Prosjekt\O10266\10266411-02\10266411-02-03 ARBEIDSOMRAADE\10266411-02 RIG\10266411-02-07 FELT- OG LABREGISTRERINGER\LAB\DSS"

excel_filer = [f for f in os.listdir(mappebane) if f.endswith(('.xls', '.xlsx', '.xlsm'))]

wb = Workbook()
ws = wb.active
ws.title = "Samlede data"

start_col = 1

# Les og lim inn data fra hver fil
for filnavn in excel_filer:
    filsti = os.path.join(mappebane, filnavn)
    try:
        df = pd.read_excel(
            filsti,
            sheet_name='Shear',
            usecols="Q,N",
            skiprows=25,
            nrows=900,
            engine='openpyxl'
        )
        df = df.dropna()
        ws.cell(row=1, column=start_col, value=f"{filnavn.split('BP ')[1]}_Q")
        ws.cell(row=1, column=start_col + 1, value=f"{filnavn.split('BP ')[1]}_N")
        for i, row in enumerate(df.itertuples(index=False), start=2):
            ws.cell(row=i, column=start_col, value=row[0])
            ws.cell(row=i, column=start_col + 1, value=row[1])
        df_gh = pd.read_excel(
            filsti,
            sheet_name='Shear',
            usecols="G,H",
            skiprows=1,
            nrows=8,
            engine='openpyxl'
        )
        df_gh = df_gh.dropna()
        ws.cell(row=1, column=start_col + 2, value=f"{filnavn.split('BP ')[1]}_G")
        ws.cell(row=1, column=start_col + 3, value=f"{filnavn.split('BP ')[1]}_H")
        for i, row in enumerate(df_gh.itertuples(index=False), start=2):
            ws.cell(row=i, column=start_col + 2, value=row[0])
            ws.cell(row=i, column=start_col + 3, value=row[1])

        start_col += 4  # 2 for hoveddata, 2 for G/H-data
    except Exception as e:
        print(f"Feil ved behandling av {filnavn}: {e}")

# Lag et scatterplot i Excel
chart = ScatterChart()
chart.title = "Samleplott av skjærdata"
chart.x_axis.title = "Horisontal skjærstyrke, t_h (kPa)"
chart.y_axis.title = "Effektiv vertikalsress, sigma_v (kPa)"
chart.x_axis.scaling.orientation = "maxMin"

# Legg til dataserier i plottet (for alle filer)
for col in range(1, start_col, 4):
    # Hoveddata
    yvalues = Reference(ws, min_col=col, min_row=2, max_row=ws.max_row)
    xvalues = Reference(ws, min_col=col+1, min_row=2, max_row=ws.max_row)
    series = Series(yvalues, xvalues, title=ws.cell(row=1, column=col).value)
    chart.series.append(series)
    # Ekstra G/H-data
    extra_y = Reference(ws, min_col=col+2, min_row=2, max_row=10)
    extra_x = Reference(ws, min_col=col+3, min_row=2, max_row=10)
    extra_series = Series(extra_y, extra_x, title=f"Tøyning {ws.cell(row=1, column=col).value}")
    extra_series.marker.symbol = "circle"
    extra_series.graphicalProperties.line.noFill = True  # No line between markers
    chart.series.append(extra_series)

ws.add_chart(chart, "A1800")

wb.save("samledata_DSS.xlsx")
print("Excel-fil med samlede data og plott er lagret som 'samledata_DSS.xlsx'")

Excel-fil med samlede data og plott er lagret som 'samledata_DSS.xlsx'
